# Pertemuan 7 — Teori Dasar: Approximate Nearest Neighbors (ANN)

Pencarian tetangga terdekat (*nearest neighbor*) adalah operasi dasar pada banyak tahap pembelajaran mesin, misalnya pada K-Means dan KNN. Cara paling sederhana untuk menemukan tetangga terdekat adalah menghitung jarak antara titik pencarian (*query point*) dengan seluruh data yang ada, lalu mengurutkannya. Cara ini disebut pencarian *exact*, dan biaya komputasinya bertambah seiring bertambahnya jumlah data serta jumlah fitur.

Sebagai gambaran, membandingkan warna kulit dengan 30 teman sekelas hanya memerlukan 30 perhitungan jarak. Membandingkan warna kulit dengan seluruh penduduk dunia memerlukan 7,8 miliar perhitungan, dan jumlahnya bertambah jika fitur yang dipakai juga bertambah, misalnya warna rambut, tinggi badan, dan berat badan. *Approximate Nearest Neighbors* (ANN) adalah pendekatan yang menjawab kebutuhan ini dengan mencari tetangga yang mendekati hasil exact, tetapi jauh lebih cepat.

Notebook ini membahas konsep dasar ANN, perhitungan metrik jarak yang dipakai, dan perbandingan tiga pustaka ANN yang digunakan pada praktikum, yaitu Annoy, FAISS, dan HNSW.

Sumber: [JS06 — ANN](https://polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-2025-2026/js06-ann-approximate-nearest-neighbors).

## 1. Ilustrasi pencarian kemiripan

Misalkan warna kulit direpresentasikan sebagai rata-rata nilai RGB dari sebuah foto. Satu orang memiliki satu nilai `avg_RGB`, dan kemiripan antar orang diukur dari jarak Euclidean antara nilai tersebut. Semakin kecil jaraknya, semakin mirip warnanya.

Contoh pada modul memakai Anto dengan `avg_RGB = (120, 65, 21)`. Sel berikut menghitung jarak Anto terhadap beberapa temannya dan mengurutkannya dari yang paling mirip.

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

anto = np.array([120, 65, 21])

teman = pd.DataFrame({
    "Nama": ["Andika", "Wendy", "Bima", "Citra", "Dewi", "Eka", "Fajar"],
    "R": [123, 132, 118, 141, 108, 126, 155],
    "G": [69, 56, 61, 70, 58, 66, 74],
    "B": [17, 15, 24, 30, 19, 22, 35],
})

rgb = teman[["R", "G", "B"]].values
teman["avg_RGB"] = [tuple(v) for v in rgb]
teman["Jarak"] = np.linalg.norm(rgb - anto, axis=1).round(2)

hasil = teman.sort_values("Jarak").reset_index(drop=True)
hasil.index = hasil.index + 1
display(hasil[["Nama", "avg_RGB", "Jarak"]])

,Nama,avg_RGB,Jarak
1,Bima,"(118, 61, 24)",5.39
2,Eka,"(126, 66, 22)",6.16
3,Andika,"(123, 69, 17)",6.40
4,Dewi,"(108, 58, 19)",14.04
5,Wendy,"(132, 56, 15)",16.16
6,Citra,"(141, 70, 30)",23.39
7,Fajar,"(155, 74, 35)",38.76


Tabel di atas merupakan hasil pencarian tetangga terdekat untuk satu query point. Proses yang sama dilakukan pada praktikum dengan jumlah data dan jumlah fitur yang jauh lebih besar.

## 2. Metrik jarak pada ANN

Metrik jarak menentukan bagaimana kedekatan dua vektor diukur. Empat metrik yang dibahas pada modul adalah Euclidean Distance, Inner Product, Cosine Similarity, dan Angular Distance. Sel berikut mengimplementasikan keempat metrik tersebut.

In [2]:
def euclidean(a, b):
    return np.linalg.norm(a - b)


def inner_product(a, b):
    return np.dot(a, b)


def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def angular(a, b):
    # Jarak sudut ternormalisasi ke rentang 0 sampai 1
    return np.arccos(np.clip(cosine_similarity(a, b), -1, 1)) / np.pi

Fungsi `angular` dihitung dari sudut antara dua vektor yang dibagi dengan $\pi$, sehingga nilainya berada pada rentang 0 sampai 1. Nilai 0 berarti arah kedua vektor identik, sedangkan nilai 1 berarti arahnya berlawanan.

In [3]:
skenario = {
    "Skenario 1 (hampir searah)": (np.array([2, 3]), np.array([2.1, 3.2])),
    "Skenario 2 (berlawanan arah)": (np.array([2, 3]), np.array([-2, -3])),
    "Skenario 3 (tegak lurus)": (np.array([2, 0]), np.array([0, 3])),
}

baris = []
for nama, (a, b) in skenario.items():
    baris.append({
        "Skenario": nama,
        "Euclidean": round(euclidean(a, b), 4),
        "Inner Product": round(inner_product(a, b), 4),
        "Cosine Similarity": round(cosine_similarity(a, b), 4),
        "Angular Distance": round(angular(a, b), 4),
    })

display(pd.DataFrame(baris))

,Skenario,Euclidean,Inner Product,Cosine Similarity,Angular Distance
0,Skenario 1 (hampir searah),0.2236,13.8,1.0,0.0023
1,Skenario 2 (berlawanan arah),7.2111,-13.0,-1.0,1.0000
2,Skenario 3 (tegak lurus),3.6056,0.0,0.0,0.5000


Hasil perhitungan memperlihatkan perilaku setiap metrik. Euclidean selalu bernilai positif dan menunjukkan jarak fisik antar titik. Inner Product dapat bernilai positif maupun negatif, dan nilai negatif menandakan arah vektor berlawanan. Cosine Similarity berada pada rentang -1 sampai 1, dengan nilai mendekati 1 berarti searah, 0 berarti tegak lurus, dan -1 berarti berlawanan arah. Angular Distance merupakan hasil transformasi cosine similarity ke skala sudut 0 sampai 1.

Pada skenario 1, cosine similarity mendekati 1 dan angular distance kecil karena kedua vektor hampir searah. Pada skenario 2, cosine similarity bernilai -1 dan angular distance bernilai 1 karena arahnya berlawanan. Pada skenario 3, cosine similarity bernilai 0 dan angular distance bernilai 0,5 karena kedua vektor tegak lurus.

## 3. Perbandingan pustaka ANN

Annoy, FAISS, dan HNSW menyelesaikan masalah yang sama dengan struktur data yang berbeda. Annoy membangun banyak pohon proyeksi acak, FAISS mengandalkan kuantisasi vektor dan pengelompokan, sedangkan HNSW memakai graf bertingkat yang dapat dinavigasi.

In [4]:
perbandingan = pd.DataFrame([
    {
        "Library": "Annoy",
        "Struktur data utama": "Random Projection Trees",
        "Metric bawaan": "Euclidean, Angular",
        "Catatan": "Sangat ringan, mudah digunakan",
    },
    {
        "Library": "FAISS",
        "Struktur data utama": "Inverted file + clustering, GPU optimized",
        "Metric bawaan": "L2, Inner Product (Cosine via normalisasi)",
        "Catatan": "Cocok untuk dataset besar dan GPU",
    },
    {
        "Library": "HNSW",
        "Struktur data utama": "Small World Graph (navigable)",
        "Metric bawaan": "L2, Inner Product, Cosine",
        "Catatan": "Lebih cepat dan akurat di CPU, fleksibel",
    },
])

display(perbandingan)

,Library,Struktur data utama,Metric bawaan,Catatan
0,Annoy,Random Projection Trees,"Euclidean, Angular","Sangat ringan, mudah digunakan"
1,FAISS,"Inverted file + clustering, GPU optimized","L2, Inner Product (Cosine via normalisasi)",Cocok untuk dataset besar dan GPU
2,HNSW,Small World Graph (navigable),"L2, Inner Product, Cosine","Lebih cepat dan akurat di CPU, fleksibel"


## Kesimpulan

Pencarian tetangga terdekat secara exact memerlukan perhitungan jarak terhadap seluruh data, sehingga biayanya meningkat tajam pada data besar dan berdimensi tinggi. ANN menjawab keterbatasan tersebut dengan mengembalikan hasil yang mendekati exact namun jauh lebih cepat. Pemilihan metrik jarak menentukan makna kedekatan yang diukur: Euclidean menekankan jarak geometris, sedangkan Cosine Similarity dan Angular Distance menekankan arah vektor. Setiap pustaka ANN memiliki karakteristik berbeda, di mana Annoy mengutamakan kesederhanaan, FAISS mengutamakan efisiensi pada data raksasa, dan HNSW menawarkan keseimbangan akurasi serta kecepatan pada CPU.